# Lab 2 · Clean, connect and check

**In class:** Part A, guided, on a messy version of the penguins data · **At home:** Part B, the same steps on your own two tables
**Due:** Sunday, November 8 at 11:59 PM (extended for Election Day) · **Submit:** your repo URL on CourseWorks

Tonight's three skills:
1. **Clean** a messy CSV with pandas and load it into DuckDB
2. **Connect** two tables with a join, on a key you've checked
3. **Check** every join by counting rows before and after

Run cells top to bottom with **Shift+Enter**.

---
# Part A · Guided: messy penguins + a lookup table

## A1. Read the messy file with pandas

`penguins_raw.csv` is the same study as last week's penguins, in its original form: long column names, extra fields, comments. **pandas** is a Python library for working with tables; a table in pandas is called a **DataFrame**.

In [2]:
from pathlib import Path
import pandas as pd
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
raw = pd.read_csv(ROOT / "data" / "penguins_raw.csv")

print(raw.shape)   # (rows, columns)
raw.head()

(344, 17)


,studyName,Sample Number,Species,Region,Island,Stage,Individual ID,Clutch Completion,Date Egg,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex,Delta 15 N (o/oo),Delta 13 C (o/oo),Comments
0,PAL0708,1,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A1,Yes,2007-11-11,39.1,18.7,181.0,3750.0,MALE,NaN,NaN,Not enough blood for isotopes.
1,PAL0708,2,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A2,Yes,2007-11-11,39.5,17.4,186.0,3800.0,FEMALE,8.94956,-24.69454,NaN
2,PAL0708,3,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A1,Yes,2007-11-16,40.3,18.0,195.0,3250.0,FEMALE,8.36821,-25.33302,NaN
3,PAL0708,4,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A2,Yes,2007-11-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Adult not sampled.
4,PAL0708,5,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N3A1,Yes,2007-11-16,36.7,19.3,193.0,3450.0,FEMALE,8.76651,-25.32426,NaN


How much is missing in each column? `isna()` marks missing values; `.sum()` counts them per column.

In [3]:
raw.isna().sum()

studyName                0
Sample Number            0
Species                  0
Region                   0
Island                   0
Stage                    0
Individual ID            0
Clutch Completion        0
Date Egg                 0
Culmen Length (mm)       2
Culmen Depth (mm)        2
Flipper Length (mm)      2
Body Mass (g)            2
Sex                     11
Delta 15 N (o/oo)       14
Delta 13 C (o/oo)       13
Comments               290
dtype: int64

Look at the species column. What would make it hard to use as a key?

In [4]:
raw["Species"].value_counts()

Species
Adelie Penguin (Pygoscelis adeliae)          152
Gentoo penguin (Pygoscelis papua)            124
Chinstrap penguin (Pygoscelis antarctica)     68
Name: count, dtype: int64

**Notice:** long names with the Latin name in parentheses, and inconsistent capitals ("Adelie **P**enguin" vs. "Gentoo **p**enguin"). Text that *looks* the same to a person often won't match exactly in a join.

## A2. Three cleaning steps

**Step 1. Rename columns** to short, lowercase names with no spaces, so they're easy to type in SQL.

In [5]:
penguins = raw.rename(columns={
    "studyName": "study",
    "Individual ID": "individual_id",
    "Species": "species_raw",
    "Island": "island",
    "Date Egg": "date_egg",
    "Clutch Completion": "clutch_completion",
    "Culmen Length (mm)": "bill_length_mm",
    "Culmen Depth (mm)": "bill_depth_mm",
    "Flipper Length (mm)": "flipper_length_mm",
    "Body Mass (g)": "body_mass_g",
    "Sex": "sex",
})
penguins.columns.tolist()

['study',
 'Sample Number',
 'species_raw',
 'Region',
 'island',
 'Stage',
 'individual_id',
 'clutch_completion',
 'date_egg',
 'bill_length_mm',
 'bill_depth_mm',
 'flipper_length_mm',
 'body_mass_g',
 'sex',
 'Delta 15 N (o/oo)',
 'Delta 13 C (o/oo)',
 'Comments']

**Step 2. Standardize text.** Make `sex` lowercase, and pull the Latin name out of the parentheses into its own column. Scientific names are written the same way everywhere, which makes them a better key than the common name.

In [6]:
penguins["sex"] = penguins["sex"].str.lower()
penguins["latin_name"] = penguins["species_raw"].str.extract(r"\((.*)\)")   # text inside ( )

penguins[["species_raw", "latin_name", "sex"]].drop_duplicates().head(6)

,species_raw,latin_name,sex
0,Adelie Penguin (Pygoscelis adeliae),Pygoscelis adeliae,male
1,Adelie Penguin (Pygoscelis adeliae),Pygoscelis adeliae,female
3,Adelie Penguin (Pygoscelis adeliae),Pygoscelis adeliae,NaN
152,Gentoo penguin (Pygoscelis papua),Pygoscelis papua,female
153,Gentoo penguin (Pygoscelis papua),Pygoscelis papua,male
178,Gentoo penguin (Pygoscelis papua),Pygoscelis papua,NaN


**Step 3. Fix types and keep what you need.** The egg date came in as text; make it a real date. Then keep only the columns we'll use.

In [7]:
penguins["date_egg"] = pd.to_datetime(penguins["date_egg"])

penguins = penguins[[
    "study", "individual_id", "latin_name", "island", "date_egg", "clutch_completion",
    "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "sex",
]]
penguins.dtypes

study                           str
individual_id                   str
latin_name                      str
island                          str
date_egg             datetime64[us]
clutch_completion               str
bill_length_mm              float64
bill_depth_mm               float64
flipper_length_mm           float64
body_mass_g                 float64
sex                             str
dtype: object

## A3. Load it into DuckDB and verify with SQL

DuckDB can read a pandas DataFrame directly by its variable name. Always check the row count after moving data: **rows in should equal rows out.**

In [8]:
con = duckdb.connect(str(ROOT / "lab2.duckdb"))
con.sql("CREATE OR REPLACE TABLE penguins_clean AS SELECT * FROM penguins")

print("rows in pandas:", len(penguins))
con.sql("SELECT count(*) AS rows_in_duckdb FROM penguins_clean")

rows in pandas: 344


┌────────────────┐
│ rows_in_duckdb │
│     int64      │
├────────────────┤
│            344 │
└────────────────┘

In [9]:
con.sql("DESCRIBE penguins_clean")

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ study             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ individual_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ latin_name        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ island            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ date_egg          │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ clutch_completion │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ bill_length_mm    │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ bill_depth_mm     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ flipper_length_mm │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ body_mass_

## A4. What is one row? Is the ID really an ID?

There's a column called `individual_id`. If it identified one penguin, every penguin would appear once per season. Let's check.

In [10]:
con.sql("""
    SELECT count(*)                      AS rows,
           count(DISTINCT individual_id) AS distinct_ids
    FROM penguins_clean
""")

┌───────┬──────────────┐
│ rows  │ distinct_ids │
│ int64 │    int64     │
├───────┼──────────────┤
│   344 │          190 │
└───────┴──────────────┘

In [11]:
con.sql("""
    SELECT study, latin_name, island, individual_id
    FROM penguins_clean
    WHERE individual_id = 'N11A1'
""")

┌─────────┬────────────────────┬─────────┬───────────────┐
│  study  │     latin_name     │ island  │ individual_id │
│ varchar │      varchar       │ varchar │    varchar    │
├─────────┼────────────────────┼─────────┼───────────────┤
│ PAL0708 │ Pygoscelis adeliae │ Biscoe  │ N11A1         │
│ PAL0809 │ Pygoscelis papua   │ Biscoe  │ N11A1         │
└─────────┴────────────────────┴─────────┴───────────────┘

**The same ID belongs to an Adélie penguin in one season and a Gentoo the next.** It can't be one bird. It's most likely a nest-and-adult label the researchers reused. So:

- `individual_id` **alone is not a key.** A column named "ID" isn't automatically unique; always check with `count(DISTINCT ...)`.
- **One row = one adult penguin at one nest in one breeding season.** That's the table's **grain**, and you should be able to say it for every table you use.

## A5. A lookup table to join

`species_lookup.csv` is a small reference table: one row per species, with its common name, Latin name, genus and group.

In [12]:
con.sql(f"CREATE OR REPLACE TABLE species AS SELECT * FROM read_csv('{ROOT / 'data' / 'species_lookup.csv'}')")
con.sql("SELECT * FROM species")

┌───────────────────┬────────────────────────┬─────────────┬───────────────────────┐
│    common_name    │       latin_name       │    genus    │      group_name       │
│      varchar      │        varchar         │   varchar   │        varchar        │
├───────────────────┼────────────────────────┼─────────────┼───────────────────────┤
│ Adélie penguin    │ Pygoscelis adeliae     │ Pygoscelis  │ brush-tailed penguins │
│ Chinstrap penguin │ Pygoscelis antarcticus │ Pygoscelis  │ brush-tailed penguins │
│ Gentoo penguin    │ Pygoscelis papua       │ Pygoscelis  │ brush-tailed penguins │
│ Emperor penguin   │ Aptenodytes forsteri   │ Aptenodytes │ great penguins        │
└───────────────────┴────────────────────────┴─────────────┴───────────────────────┘

Before joining, check that the key is **unique** in the lookup table. If a key appeared twice, every matching penguin would be copied twice.

In [13]:
con.sql("""
    SELECT latin_name, count(*) AS n
    FROM species
    GROUP BY latin_name
    HAVING count(*) > 1
""")   # no rows back = the key is unique

┌────────────┬───────┐
│ latin_name │   n   │
│  varchar   │ int64 │
└────────────┴───────┘
        0 rows      

## A6. Join, and count what comes out

`JOIN ... ON` lines up rows where the key matches. We expect **344** rows out, one per penguin row.

In [14]:
con.sql("""
    SELECT count(*) AS rows_after_join
    FROM penguins_clean p
    JOIN species s ON p.latin_name = s.latin_name
""")

┌─────────────────┐
│ rows_after_join │
│      int64      │
├─────────────────┤
│             276 │
└─────────────────┘

**Only 276, not 344.** An inner `JOIN` silently drops rows that don't match. Nothing errored. This is a data cascade in the making.

## A7. Find what didn't match with a LEFT JOIN

A `LEFT JOIN` keeps every penguin row. Where there's no match, the lookup's columns come back empty (`NULL`), so we can find the problem rows.

In [15]:
con.sql("""
    SELECT p.latin_name, count(*) AS unmatched_rows
    FROM penguins_clean p
    LEFT JOIN species s ON p.latin_name = s.latin_name
    WHERE s.latin_name IS NULL
    GROUP BY p.latin_name
""")

┌───────────────────────┬────────────────┐
│      latin_name       │ unmatched_rows │
│        varchar        │     int64      │
├───────────────────────┼────────────────┤
│ Pygoscelis antarctica │             68 │
└───────────────────────┴────────────────┘

Compare with the lookup: the data says **Pygoscelis antarctic*a***, the lookup says **Pygoscelis antarctic*us***. Two sources spelled the same species differently, and 68 penguins vanished from the join.

## A8. Fix the key, then join again

Fix it at the source of the problem (the cleaning step), then reload and rejoin.

In [16]:
penguins["latin_name"] = penguins["latin_name"].replace(
    {"Pygoscelis antarctica": "Pygoscelis antarcticus"}
)
con.sql("CREATE OR REPLACE TABLE penguins_clean AS SELECT * FROM penguins")

con.sql("""
    SELECT count(*) AS rows_after_join
    FROM penguins_clean p
    JOIN species s ON p.latin_name = s.latin_name
""")   # now 344

┌─────────────────┐
│ rows_after_join │
│      int64      │
├─────────────────┤
│             344 │
└─────────────────┘

Which way you join matters. Start from the **lookup** with a `LEFT JOIN` to see species that have no penguins:

In [17]:
con.sql("""
    SELECT s.common_name, count(p.latin_name) AS penguin_rows
    FROM species s
    LEFT JOIN penguins_clean p ON p.latin_name = s.latin_name
    GROUP BY s.common_name
    ORDER BY penguin_rows DESC
""")

┌───────────────────┬──────────────┐
│    common_name    │ penguin_rows │
│      varchar      │    int64     │
├───────────────────┼──────────────┤
│ Adélie penguin    │          152 │
│ Gentoo penguin    │          124 │
│ Chinstrap penguin │           68 │
│ Emperor penguin   │            0 │
└───────────────────┴──────────────┘

## A9. What a duplicate key does

Here we fake a lookup table that lists Gentoo twice, then join. Watch the row count.

In [18]:
con.sql("""
    WITH species_with_dupe AS (
        SELECT * FROM species
        UNION ALL
        SELECT * FROM species WHERE common_name = 'Gentoo penguin'
    )
    SELECT count(*) AS rows_after_join
    FROM penguins_clean p
    JOIN species_with_dupe s ON p.latin_name = s.latin_name
""")   # 468: every Gentoo row was copied

┌─────────────────┐
│ rows_after_join │
│      int64      │
├─────────────────┤
│             468 │
└─────────────────┘

**344 became 468**, and every total or average involving Gentoos would now be wrong. That's why you check key uniqueness (A5) before every join.

## A10. Use the join to answer a question

Now that the tables connect, ask a question that needs both: the lookup's readable names, the penguin table's measurements.

In [19]:
con.sql("""
    SELECT s.common_name,
           p.island,
           count(*)                    AS penguins,
           round(avg(p.body_mass_g))   AS avg_mass_g
    FROM penguins_clean p
    JOIN species s ON p.latin_name = s.latin_name
    GROUP BY s.common_name, p.island
    ORDER BY s.common_name, p.island
""")

┌───────────────────┬───────────┬──────────┬────────────┐
│    common_name    │  island   │ penguins │ avg_mass_g │
│      varchar      │  varchar  │  int64   │   double   │
├───────────────────┼───────────┼──────────┼────────────┤
│ Adélie penguin    │ Biscoe    │       44 │     3710.0 │
│ Adélie penguin    │ Dream     │       56 │     3688.0 │
│ Adélie penguin    │ Torgersen │       52 │     3706.0 │
│ Chinstrap penguin │ Dream     │       68 │     3733.0 │
│ Gentoo penguin    │ Biscoe    │      124 │     5076.0 │
└───────────────────┴───────────┴──────────┴────────────┘

### The join checklist (use it in Part B)

1. **Grain:** what is one row in each table?
2. **Key:** which column connects them, and is it **unique** in the lookup side?
3. **Clean the key** the same way on both sides (spelling, capitals, spaces, codes vs. names, types).
4. **Count rows** before and after. More rows = a duplicate key. Fewer = unmatched keys.
5. **LEFT JOIN … IS NULL** to see exactly what didn't match, then decide: fix, drop, or keep as unknown.

---
# Part B · At home: your own two tables

Apply the checklist to the two tables you chose in Lab 1. If you'd like to swap datasets, that's fine; you just need two public tables that share a column.

## B1. Load both tables with pandas

In [20]:
FILE_A = "ems_2023_10.csv"     # <- your main table's file in data/
FILE_B = "acs_income_2023.csv"   # <- the related table's file in data/

a = pd.read_csv(ROOT / "data" / FILE_A)
b = pd.read_csv(ROOT / "data" / FILE_B)
print("A:", a.shape, "  B:", b.shape)
a.head()

A: (141460, 31)   B: (206, 4)


,incident_id,incident_datetime,initial_call_type,initial_severity_level_code,final_call_type,final_severity_level_code,first_assignment_datetime,valid_dispatch_rspns_time_indc,dispatch_response_seconds_qy,first_activation_datetime,...,zipcode,policeprecinct,citycouncildistrict,communitydistrict,communityschooldistrict,congressionaldistrict,reopen_indicator,special_event_indicator,standby_indicator,transfer_indicator
0,78327772,2023-10-01T00:00:17.000,STATEP,2,STATEP,2,2023-10-01T00:00:37.000,Y,20,2023-10-01T00:01:23.000,...,10467.0,49.0,15.0,211.0,11.0,14.0,N,N,N,N
1,78327773,2023-10-01T00:00:56.000,STNDBY,8,STNDBY,8,2023-10-01T07:41:02.000,Y,27606,2023-10-01T07:41:02.000,...,10022.0,18.0,4.0,105.0,2.0,12.0,Y,N,Y,N
2,78327775,2023-10-01T00:01:09.000,DRUG,4,DRUG,4,2023-10-01T00:02:43.000,Y,94,2023-10-01T00:02:54.000,...,10016.0,13.0,2.0,106.0,2.0,12.0,N,N,N,N
3,78327776,2023-10-01T00:01:22.000,DRUG,4,DRUG,4,2023-10-01T00:02:07.000,Y,45,2023-10-01T00:03:28.000,...,10453.0,46.0,14.0,205.0,9.0,15.0,N,N,N,N
4,78327777,2023-10-01T00:01:36.000,SICK,6,SICK,6,2023-10-01T00:01:36.000,Y,0,2023-10-01T00:01:36.000,...,10457.0,44.0,16.0,204.0,9.0,15.0,N,N,N,N


In [21]:
b.head()

,zcta,GEO_ID,B19013_E001,B19013_M001
0,10001,860Z200US10001,123393,22852
1,10002,860Z200US10002,46525,5561
2,10003,860Z200US10003,153750,11191
3,10004,860Z200US10004,220592,53147
4,10005,860Z200US10005,211810,29291


**Grain** (double-click to edit). One row in table A is: one EMS incident recorded in October 2023. 

One row in table B is: one selected Census ZIP Code Tabulation Area (ZCTA), with its median household income estimate and margin of error from the 2019–2023 ACS five-year estimates.

## B2. Clean: at least three steps

Use the Part A steps as a model: rename columns, standardize text, fix types, drop columns you don't need. Write each step in its own line with a comment saying *why*.

In [28]:
# Step 1: standardize column names and rename income fields for readable SQL
a.columns = a.columns.str.strip().str.lower()
b.columns = b.columns.str.strip().str.lower()
b = b.rename(columns={"b19013_e001": "median_household_income", "b19013_m001": "income_moe"})

# Step 2: convert both geographic keys to text and remove numeric-format suffixes
a["zipcode"] = a["zipcode"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
b["zcta"] = b["zcta"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)

# Step 3: convert incident timestamps from text to datetime
a["incident_datetime"] = pd.to_datetime(a["incident_datetime"])

# Step 4: replace Census special codes with missing values, keeping all rows
for col in ["median_household_income", "income_moe"]:
    b[col] = pd.to_numeric(b[col]).astype("Int64")
    b[col] = b[col].mask(b[col].isin([-666666666, -222222222, -333333333]))

a.head()

,incident_id,incident_datetime,initial_call_type,initial_severity_level_code,final_call_type,final_severity_level_code,first_assignment_datetime,valid_dispatch_rspns_time_indc,dispatch_response_seconds_qy,first_activation_datetime,...,zipcode,policeprecinct,citycouncildistrict,communitydistrict,communityschooldistrict,congressionaldistrict,reopen_indicator,special_event_indicator,standby_indicator,transfer_indicator
0,78327772,2023-10-01 00:00:17,STATEP,2,STATEP,2,2023-10-01T00:00:37.000,Y,20,2023-10-01T00:01:23.000,...,10467,49.0,15.0,211.0,11.0,14.0,N,N,N,N
1,78327773,2023-10-01 00:00:56,STNDBY,8,STNDBY,8,2023-10-01T07:41:02.000,Y,27606,2023-10-01T07:41:02.000,...,10022,18.0,4.0,105.0,2.0,12.0,Y,N,Y,N
2,78327775,2023-10-01 00:01:09,DRUG,4,DRUG,4,2023-10-01T00:02:43.000,Y,94,2023-10-01T00:02:54.000,...,10016,13.0,2.0,106.0,2.0,12.0,N,N,N,N
3,78327776,2023-10-01 00:01:22,DRUG,4,DRUG,4,2023-10-01T00:02:07.000,Y,45,2023-10-01T00:03:28.000,...,10453,46.0,14.0,205.0,9.0,15.0,N,N,N,N
4,78327777,2023-10-01 00:01:36,SICK,6,SICK,6,2023-10-01T00:01:36.000,Y,0,2023-10-01T00:01:36.000,...,10457,44.0,16.0,204.0,9.0,15.0,N,N,N,N


In [29]:
b[["median_household_income", "income_moe"]].isna().sum()

median_household_income    23
income_moe                 26
dtype: int64

## B3. Load into DuckDB and check the key

In [31]:
con = duckdb.connect(str(ROOT / "lab2.duckdb"))

con.sql("CREATE OR REPLACE TABLE ems AS SELECT * FROM a")
con.sql("CREATE OR REPLACE TABLE acs AS SELECT * FROM b")

KEY_A = "zipcode"
KEY_B = "zcta"

# Check that loading preserved all rows
print("EMS — pandas:", len(a), "DuckDB:", con.sql("SELECT count(*) FROM ems").fetchone()[0])
print("ACS — pandas:", len(b), "DuckDB:", con.sql("SELECT count(*) FROM acs").fetchone()[0])

# Check whether any ZCTA appears more than once in the income table
con.sql(f"""
    SELECT "{KEY_B}", count(*) AS n
    FROM acs
    GROUP BY "{KEY_B}"
    HAVING count(*) > 1
    LIMIT 10
""")

EMS — pandas: 141460 DuckDB: 141460
ACS — pandas: 206 DuckDB: 206


┌─────────┬───────┐
│  zcta   │   n   │
│ varchar │ int64 │
└─────────┴───────┘
      0 rows     

## B4. Join and count

In [32]:
print("rows in table A:", con.sql("SELECT count(*) FROM table_a").fetchone()[0])

con.sql(f"""
    SELECT count(*) AS rows_after_join
    FROM table_a a
    JOIN table_b b ON a."{KEY_A}" = b."{KEY_B}"
""")

rows in table A: 141460


┌─────────────────┐
│ rows_after_join │
│      int64      │
├─────────────────┤
│          139845 │
└─────────────────┘

## B5. Find what didn't match

In [34]:
con.sql(f"""
    SELECT a."{KEY_A}", count(*) AS unmatched_rows
    FROM table_a a
    LEFT JOIN table_b b ON a."{KEY_A}" = b."{KEY_B}"
    WHERE b."{KEY_B}" IS NULL
    GROUP BY a."{KEY_A}"
    ORDER BY unmatched_rows DESC
    LIMIT 20
""")

┌─────────┬────────────────┐
│ zipcode │ unmatched_rows │
│ double  │     int64      │
├─────────┼────────────────┤
│    NULL │           1487 │
│ 10121.0 │             52 │
│ 10000.0 │             26 │
│ 10106.0 │              9 │
│ 11251.0 │              7 │
│ 10118.0 │              7 │
│ 10120.0 │              5 │
│ 10281.0 │              4 │
│ 10107.0 │              2 │
│ 11242.0 │              2 │
│ 11695.0 │              2 │
│ 10041.0 │              2 │
│ 10048.0 │              2 │
│ 10179.0 │              2 │
│ 10176.0 │              1 │
│ 10178.0 │              1 │
│ 10123.0 │              1 │
│ 12345.0 │              1 │
│ 10105.0 │              1 │
│    83.0 │              1 │
└─────────┴────────────────┘
  20 rows        2 columns

**What didn't match, and what will you do about it?** (fix the key, drop those rows, or keep them as unknown, and why)

A total of 1,615 EMS records did not match the ACS table: 1,487 had missing ZIP codes, and 128 had ZIP codes outside the 206 ZCTAs in the ACS subset. These 128 records contained 19 distinct unmatched codes, including the malformed code “83.” ZIP codes and Census ZCTAs are not equivalent geographic units, so I would not force matches by changing codes without verification. I would retain unmatched records with unknown income in the combined dataset using a LEFT JOIN, but exclude them from comparisons that require an income estimate and report that exclusion.



## B6. Answer one question that needs both tables

Write a query that joins the tables and uses columns from **both**, e.g. average GDP by region, complaints per 1,000 residents by ZIP code.

In [37]:
con.sql(f"""
    SELECT
        CASE
            WHEN b.median_household_income < 50000 THEN '1. Below $50,000'
            WHEN b.median_household_income < 100000 THEN '2. $50,000–$99,999'
            ELSE '3. $100,000 or more'
        END AS income_group,
        count(*) AS incidents,
        count(DISTINCT b."{KEY_B}") AS zctas,
        round(avg(a.incident_response_seconds_qy) / 60.0, 2) AS avg_response_minutes,
        round(median(a.incident_response_seconds_qy) / 60.0, 2) AS median_response_minutes
    FROM ems a
    JOIN acs b ON a."{KEY_A}" = b."{KEY_B}"
    WHERE b.median_household_income IS NOT NULL
      AND a.valid_incident_rspns_time_indc = 'Y'
      AND a.incident_response_seconds_qy IS NOT NULL
      AND a.incident_response_seconds_qy >= 0
    GROUP BY income_group
    ORDER BY income_group
""")

┌─────────────────────┬───────────┬───────┬──────────────────────┬─────────────────────────┐
│    income_group     │ incidents │ zctas │ avg_response_minutes │ median_response_minutes │
│       varchar       │   int64   │ int64 │        double        │         double          │
├─────────────────────┼───────────┼───────┼──────────────────────┼─────────────────────────┤
│ 1. Below $50,000    │     33120 │    23 │                15.64 │                    9.18 │
│ 2. $50,000–$99,999  │     66829 │    92 │                12.73 │                    8.57 │
│ 3. $100,000 or more │     31864 │    68 │                 13.3 │                     8.4 │
└─────────────────────┴───────────┴───────┴──────────────────────┴─────────────────────────┘

**What does the result tell you?** (one or two sentences)

Areas with median household incomes below $50,000 had the longest average EMS response time (15.64 minutes), compared with 12.73 minutes for middle-income areas and 13.30 minutes for high-income areas. This comparison shows an association, not a causal effect of income.

## B7. Submit

1. Run the cell below.
2. Answer the two reflection questions in `labs/lab2.md`.
3. Save this notebook (Cmd+S), then commit and push: **Source Control** → `Lab 2` → **Commit** → **Sync Changes**.
4. Paste your repo URL into the Lab 2 assignment on CourseWorks.

In [39]:
con.close()
print("Done. Save, commit, push, and submit your repo URL.")

Done. Save, commit, push, and submit your repo URL.
